# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alemjarebica-cloud/ML-flyrank-AlemJ/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print("Konektovano.")

Konektovano.


One row = one (report date × client × content item) observation in fact_content_daily_performance - one content item's search performance for one day, for one client. Time window: month=2026-03 (mid-panel), split at day 15: days 1–15 = knowable window, days 16–end = outcome window.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rel = "hf://datasets/FlyRank/internship-warehouse"
DATE_COL         = "report_date"
CLIENT_COL       = "client_hash_id"
CONTENT_COL      = "content_hash_id"
POSITION_COL     = "gsc_avg_position"
CLICKS_COL       = "gsc_clicks"
IMPRESSIONS_COL  = "gsc_impressions"
AVAILABILITY_COL = "gsc_data_available"
MONTH = "2026-03"

sample = con.sql(f"""
    SELECT {CONTENT_COL}, {CLIENT_COL}, {DATE_COL}
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    LIMIT 1
""").df().iloc[0]

con.sql(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE {CONTENT_COL}='{sample[CONTENT_COL]}' AND {CLIENT_COL}='{sample[CLIENT_COL]}' AND {DATE_COL}=DATE '{sample[DATE_COL]}'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows
0,1


In [18]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT {CONTENT_COL}) AS n_content,
           COUNT(DISTINCT {CLIENT_COL}) AS n_clients,
           MIN({DATE_COL}) AS first_date, MAX({DATE_COL}) AS last_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content,n_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [19]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE {AVAILABILITY_COL} IS TRUE) AS available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


Features (days 1–15, from fact_content_daily_performance):
- avg_position_h1 (avg of gsc_avg_position)
- total_clicks_h1 (sum of gsc_clicks)
- total_impressions_h1 (sum of gsc_impressions)
- ctr_h1 (clicks / impressions)
- zero_impression_days_h1 (count of days with gsc_impressions = 0)

Label (proxy): is_declining = 1 if avg_position in days 16–end is worse (higher) than avg_position_h1, else 0

Context: dim_content (content metadata, joined for description only, doesn't change grain); dim_clients (checked only for panel coverage, not joined into features)

Excluded: fact_content_query_90d — it's a floating 90-day window (last-30/prev-30 sub-windows) that doesn't align to the month=2026-03 partition boundary, so including it risks mixing in information from outside this decision window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT {CONTENT_COL}) AS n_content
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content
0,519606,519606


Grain: confirmed, 1 row per (content, client, date) triple.
Counts/window: month=2026-03 → 9,841,378 rows, 331,437 content items, 55 clients, 2026-03-01 to 2026-03-31.
Availability: 3,611,061 / 9,841,378 rows (37%) have gsc_data_available IS TRUE.
Missing values: checked below.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE {POSITION_COL} IS NULL) AS null_position,
        COUNT(*) FILTER (WHERE {CLICKS_COL} IS NULL) AS null_clicks,
        COUNT(*) FILTER (WHERE {IMPRESSIONS_COL} IS NULL) AS null_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,null_position,null_clicks,null_impressions
0,6230317,0,0


- Position (gsc_avg_position) is NULL for ~63% of rows — exactly where gsc_data_available is not TRUE. This data can't tell you ranking position for a page/day GSC didn't report on, even though clicks/impressions columns are never null.
- Unbalanced panel: 55 clients, but dim_clients shows different gsc_data_start/ga4_data_start per client, an early month for one client may be missing entirely for another, so cross-client averages aren't apples-to-apples.
- This is a proxy label (half-month vs half-month), not ground truth, a 15-day window is noisy and can't tell you a page's true underlying trend, only a short-term direction.
- AI-referral columns (ai_chatgpt, ai_perplexity, etc.) are recent additions to search behavior, early months in the panel likely can't tell you anything meaningful there since the behavior itself didn't exist yet.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT
        {AVAILABILITY_COL},
        COUNT(*) AS n_rows,
        COUNT(*) FILTER (WHERE {POSITION_COL} IS NULL) AS null_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY {AVAILABILITY_COL}
""").df()

,gsc_data_available,n_rows,null_position
0,False,6230317,6230317
1,True,3611061,0


In [23]:
con.sql(f"""
    SELECT {CLIENT_COL}, COUNT(DISTINCT {DATE_COL}) AS days_in_march
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY {CLIENT_COL}
    ORDER BY days_in_march ASC
    LIMIT 5
""").df()

,client_hash_id,days_in_march
0,client_e00b29e582949543,9
1,client_810019792c9b8efc,12
2,client_f6f0cdf26d03d7bd,13
3,client_86ebc2f12c01f586,29
4,client_d211cb07b9059bab,31


In [24]:
feat = con.sql(f"""
    SELECT {CONTENT_COL} AS content_id, {CLIENT_COL} AS client_id,
        AVG({POSITION_COL}) AS avg_position_h1,
        SUM({CLICKS_COL}) AS total_clicks_h1,
        SUM({IMPRESSIONS_COL}) AS total_impressions_h1,
        SUM({CLICKS_COL})*1.0/NULLIF(SUM({IMPRESSIONS_COL}),0) AS ctr_h1,
        COUNT(*) FILTER (WHERE {IMPRESSIONS_COL}=0) AS zero_impression_days_h1
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE {DATE_COL} <= DATE '{MONTH}-15' AND {POSITION_COL} IS NOT NULL
    GROUP BY {CONTENT_COL}, {CLIENT_COL}
""").df()
feat.head()

,content_id,client_id,avg_position_h1,total_clicks_h1,total_impressions_h1,ctr_h1,zero_impression_days_h1
0,content_ac8663da7484669a,client_62f4a7e64f5e0096,3.597222,0.0,20.0,0.000000,0
1,content_39d7361b4945d504,client_62f4a7e64f5e0096,3.659683,0.0,57.0,0.000000,0
2,content_d49a012dcb924e31,client_62f4a7e64f5e0096,4.520919,0.0,246.0,0.000000,0
3,content_614baf2af4330bd7,client_62f4a7e64f5e0096,4.390322,1.0,413.0,0.002421,0
4,content_225dc9235023be5f,client_62f4a7e64f5e0096,9.961993,1.0,279.0,0.003584,0


In [25]:
h2 = con.sql(f"""
    SELECT {CONTENT_COL} AS content_id, {CLIENT_COL} AS client_id,
        AVG({POSITION_COL}) AS avg_position_h2
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE {DATE_COL} > DATE '{MONTH}-15' AND {POSITION_COL} IS NOT NULL
    GROUP BY {CONTENT_COL}, {CLIENT_COL}
""").df()

labeled = feat.merge(h2, on=["content_id","client_id"])
labeled["is_declining"] = (labeled["avg_position_h2"] > labeled["avg_position_h1"]).astype(int)
labeled["is_declining"].value_counts()

,count
is_declining,
1,76484
0,64983


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

honest = ["avg_position_h1","total_clicks_h1","total_impressions_h1","ctr_h1","zero_impression_days_h1"]

def score(cols):
    X = labeled[cols].fillna(0); y = labeled["is_declining"]
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.3,random_state=0,stratify=y)
    clf = DecisionTreeClassifier(max_depth=4,random_state=0).fit(Xtr,ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:,1])

honest_auc = score(honest)
leaky_auc = score(honest + ["avg_position_h2"])
print(f"Honest AUC: {honest_auc:.3f}")
print(f"Leaky AUC (+avg_position_h2): {leaky_auc:.3f}  jump: {leaky_auc-honest_auc:+.3f}")
print(f"Keeping honest score: {honest_auc:.3f}")

Honest AUC: 0.658
Leaky AUC (+avg_position_h2): 0.867  jump: +0.209
Keeping honest score: 0.658


Leak: avg_position_h2 is dated after the decision moment (it's literally what is_declining is computed from). Adding it let the model "see the answer," jumping AUC from 0.657 to 0.872. That's not a better model, it's a label leak. Removed honest score (0.657, five h1-only features) is what's kept.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.